In [0]:
%run ../utils/utils_config


In [0]:
# this gives you everything the _delta_log contains — in readable form
spark.sql("""
    DESCRIBE HISTORY workspace.wind_turbine.bronze
""").select(
    "version",
    "timestamp",
    "operation",
    "operationParameters",
    "operationMetrics"
).show(truncate=False)

In [0]:
# drill into one version's metrics
history = spark.sql("DESCRIBE HISTORY workspace.wind_turbine.bronze").collect()

for row in history:
    print(f"Version   : {row['version']}")
    print(f"Operation : {row['operation']}")
    print(f"Metrics   : {row['operationMetrics']}")
    print(f"Timestamp : {row['timestamp']}")
    print("---")

In [0]:
# equivalent of sp_help in SQL Server
spark.sql("DESCRIBE TABLE workspace.wind_turbine.bronze").show(truncate=False)

In [0]:
# more detail — physical metadata
spark.sql("DESCRIBE DETAIL workspace.wind_turbine.bronze").show(truncate=False)

In [0]:
# the audit trail — every operation ever run on this table
spark.sql("DESCRIBE HISTORY workspace.wind_turbine.bronze").show(truncate=False)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

# version 1 — append some extra rows (simulate a second file arriving)
df_extra = spark.sql("""
    SELECT * FROM workspace.wind_turbine.bronze LIMIT 5
""").withColumn("timestamp_utc", lit("2026-06-04T03:00:00Z")) \
   .withColumn("active_power_kw", lit(999.99))

df_extra.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.wind_turbine.bronze")

print("Version 1 created — appended 5 rows")

In [0]:
# version 2 — update one row (copy-on-write kicks in)
spark.sql("""
    UPDATE workspace.wind_turbine.bronze
    SET active_power_kw = -1.0
    WHERE asset_id = 'WTG_01'
    AND timestamp_utc = '2026-06-04T03:00:00Z'
""")
print("Version 2 created — updated WTG_01 row")

In [0]:
# version 3 — delete the test rows we added
spark.sql("""
    DELETE FROM workspace.wind_turbine.bronze
    WHERE timestamp_utc = '2026-06-04T03:00:00Z'
""")
print("Version 3 created — deleted test rows")

In [0]:
# now check history — you should see 4 versions (0,1,2,3)
spark.sql("DESCRIBE HISTORY workspace.wind_turbine.bronze") \
     .select("version","timestamp","operation","operationParameters") \
     .show(truncate=False)

In [0]:
# current state — should be back to 60 rows, no WTG_01 test row
current = spark.sql("SELECT COUNT(*) as rows FROM workspace.wind_turbine.bronze").collect()[0][0]
print(f"Current rows: {current}")

# version 0 — your original Day 2 load
v0 = spark.sql("SELECT COUNT(*) as rows FROM workspace.wind_turbine.bronze VERSION AS OF 0").collect()[0][0]
print(f"Version 0 rows: {v0}")

# version 1 — after append (should be 65 rows)
v1 = spark.sql("SELECT COUNT(*) as rows FROM workspace.wind_turbine.bronze VERSION AS OF 1").collect()[0][0]
print(f"Version 1 rows: {v1}")

# version 2 — after update
v2 = spark.sql("""
    SELECT asset_id, timestamp_utc, active_power_kw 
    FROM workspace.wind_turbine.bronze VERSION AS OF 2
    WHERE timestamp_utc = '2026-06-04T03:00:00Z'
""")
print("Version 2 — WTG_01 test row:")
v2.show(truncate=False)

In [0]:
# query by timestamp instead of version number
spark.sql("""
    SELECT COUNT(*) as rows
    FROM workspace.wind_turbine.bronze
    TIMESTAMP AS OF '2026-06-14'
""").show()

In [0]:
# simulate a disaster — delete all rows
spark.sql("DELETE FROM workspace.wind_turbine.bronze WHERE 1=1")

count_after = spark.sql("SELECT COUNT(*) as rows FROM workspace.wind_turbine.bronze").collect()[0][0]
print(f"After accidental delete: {count_after} rows")

In [0]:
# restore to version 0 — original clean state
spark.sql("RESTORE TABLE workspace.wind_turbine.bronze TO VERSION AS OF 0")

count_restored = spark.sql("SELECT COUNT(*) as rows FROM workspace.wind_turbine.bronze").collect()[0][0]
print(f"After restore: {count_restored} rows")

In [0]:
# check history — RESTORE appears as a new version
spark.sql("DESCRIBE HISTORY workspace.wind_turbine.bronze") \
     .select("version","operation","operationParameters") \
     .show(truncate=False)

In [0]:
# before optimize — how many files?
before = spark.sql("DESCRIBE DETAIL workspace.wind_turbine.bronze").collect()[0]
print(f"Files before OPTIMIZE: {before['numFiles']}")
print(f"Size before OPTIMIZE : {before['sizeInBytes']} bytes")

In [0]:
# optimize — compacts small files, improves query performance
spark.sql("OPTIMIZE workspace.wind_turbine.bronze")
print("OPTIMIZE complete")

In [0]:
# after optimize — fewer, larger files
after = spark.sql("DESCRIBE DETAIL workspace.wind_turbine.bronze").collect()[0]
print(f"Files after OPTIMIZE: {after['numFiles']}")
print(f"Size after OPTIMIZE : {after['sizeInBytes']} bytes")